In [1]:
import sys
import ibis
sys.path.append("../")

from sql_ai_agent.SqlAgent import SqlAgent


## Model Settings

In [35]:
base_url = "http://model-runner.docker.internal/engines/v1"
api_key = "docker"
temperature = 0
model = "ai/llama3.2:latest"
# model = "ai/devstral-small:24B"
model = "ai/granite-4.0-h-micro"
# model = "ai/gemma3n"
fallback_model = "ai/gemma3n"
fallback_model = "ai/devstral-small:24B"
fallback_model = "ai/granite-4.0-h-micro"
tbl_name = "air_traffic"
max_token = 10000


In [28]:
con_ibis = ibis.postgres.connect(
    user="postgres",
    password="password",
    host="postgres",
    port=5432,
    database="my_db",
)


In [36]:
agent = SqlAgent(
    api_key=api_key,
    base_url=base_url,
    model=model,
    con=con_ibis,
    fallback=True,
    fallback_model=fallback_model,
    tbl_name=tbl_name,
)


In [5]:
print(agent.character_distinct_values_reformated)

The following columns have known categorical values:
- "Operating Airline": 'ABC Aerolineas S.A. de C.V. dba Interjet', 'Aer Lingus, Ltd.', 'Aeroflot Russian International Airlines', 'Aeromexico', 'Air 2000', 'Air Atlanta Icelandic', 'Air Berlin', 'Air Canada', 'Air Canada Jazz', 'Air China' ...
- "Operating Airline IATA Code": '4O', '4T', '5Y', '9W', 'A8', 'AA', 'AB', 'AC', 'AF', 'AI' ...
- "Published Airline": 'ABC Aerolineas S.A. de C.V. dba Interjet', 'Aer Lingus, Ltd.', 'Aeroflot Russian International Airlines', 'Aeromexico', 'Air 2000', 'Air Atlanta Icelandic', 'Air Berlin', 'Air Canada', 'Air China', 'Air Europe' ...
- "Published Airline IATA Code": '4O', '4T', '5Y', '9W', 'A8', 'AA', 'AB', 'AC', 'AF', 'AI' ...
- "GEO Summary": 'Domestic', 'International'
- "GEO Region": 'Asia', 'Australia / Oceania', 'Canada', 'Central America', 'Europe', 'Mexico', 'Middle East', 'South America', 'US'
- "Activity Type Code": 'Deplaned', 'Enplaned', 'Thru / Transit'
- "Price Category Code": 'Low

In [21]:
question = "How many rows are in the dataset?"


agent.ask_question(question=question, verbose=False)



QueryOutput(success=True, query='SELECT COUNT(*) FROM "air_traffic"', data=   count
0  38546, error=None)

In [30]:
question = "Group the air passenger by the date and return the months with more than 0 passengers?"


agent.ask_question(question=question, verbose=False)


QueryOutput(success=True, query='SELECT\n  EXTRACT(MONTH FROM "Date") AS "month"\nFROM "air_traffic"\nGROUP BY\n  "month"\nHAVING\n  COUNT("Passenger Count") > 0;', data=   month
0      3
1      1
2      7
3     11
4      4
5      9
6      6
7   1E+1
8      8
9      2
10    12
11     5, error=None)

In [31]:
question = "Group the air passenger by the date and return the months with more than 0 passengers?"


agent.ask_question(question=question, verbose=False, trials = 5)


QueryOutput(success=True, query='SELECT\n  EXTRACT(MONTH FROM "Date") AS "month"\nFROM "air_traffic"\nGROUP BY\n  "month"\nHAVING\n  COUNT("Passenger Count") > 0;', data=   month
0      3
1      1
2      7
3     11
4      4
5      9
6      6
7   1E+1
8      8
9      2
10    12
11     5, error=None)

In [34]:
question = "Return all observations for the month with positive number of passengers"
a =agent.ask_question(question=question, verbose=False)

print(a.query)

print(a.data)


Error in the query processing, trying to debug...
Trial:  3
column "date" does not exist
LINE 1: ...(MONTH FROM "Date") IN (SELECT EXTRACT(MONTH FROM date) FROM...
                                                             ^
HINT:  Perhaps you meant to reference the column "air_traffic.Date" or the column "air_traffic.Date".
SELECT "Year", "Date", "Passenger Count"
FROM "air_traffic"
WHERE EXTRACT(MONTH FROM "Date") IN (SELECT EXTRACT(MONTH FROM "Date") FROM air_traffic WHERE "Passenger Count" > 0);

       Year       Date  Passenger Count
0      1999 1999-07-01            31432
1      1999 1999-07-01            31353
2      1999 1999-07-01             2518
3      1999 1999-07-01             1324
4      1999 1999-07-01             1198
...     ...        ...              ...
38541  2025 2025-07-01            13178
38542  2025 2025-07-01            14451
38543  2025 2025-07-01            12475
38544  2025 2025-07-01             8602
38545  2025 2025-07-01             5679

[38546 rows

In [10]:
a.data

,Year,Date,Operating Airline,Operating Airline IATA Code,Published Airline,Published Airline IATA Code,GEO Summary,GEO Region,Activity Type Code,Price Category Code,Terminal,Boarding Area,Passenger Count
0,2000,2000-01-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Deplaned,Low Fare,Terminal 1,B,17578
1,2000,2000-01-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Enplaned,Low Fare,Terminal 1,B,16606
2,2000,2000-01-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Thru / Transit,Low Fare,Terminal 1,B,2616
3,2000,2000-01-01,ATA Airlines,TZ,ATA Airlines,TZ,International,Canada,Deplaned,Low Fare,Terminal 1,B,95
4,2000,2000-01-01,Aeroflot Russian International Airlines,None,Aeroflot Russian International Airlines,None,International,Europe,Deplaned,Other,Terminal 2,D,811
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3173,2025,2025-01-01,Virgin Atlantic,VS,Virgin Atlantic,VS,International,Europe,Enplaned,Other,International,A,5829
3174,2025,2025-01-01,WestJet,WS,WestJet,WS,International,Canada,Deplaned,Other,International,A,2229
3175,2025,2025-01-01,WestJet,WS,WestJet,WS,International,Canada,Enplaned,Other,International,A,2105
3176,2025,2025-01-01,ZIPAIR Tokyo Inc,ZG,ZIPAIR Tokyo Inc,ZG,International,Asia,Deplaned,Other,International,A,8421


In [25]:
agent.ask_question(
    question="What are the unique values of the Activity Type Code field?"
)


QueryOutput(success=True, query='SELECT DISTINCT "Activity Type Code" FROM air_traffic;', data=  Activity Type Code
0           Deplaned
1     Thru / Transit
2           Enplaned, error=None)

In [37]:
additional_context = ""
question = "What are the unique values of the Activity Type Code field?"
agent.ask_question(question=question, additional_context=additional_context, trials=3)


QueryOutput(success=True, query='SELECT DISTINCT "Activity Type Code"\nFROM "air_traffic";', data=  Activity Type Code
0           Deplaned
1     Thru / Transit
2           Enplaned, error=None)

In [38]:
additional_context = ""
question = "How many passengers landed at the airport in 2019?"
a = agent.ask_question(question=question, additional_context=additional_context, trials=5)

print(a.query)
print(a.data)


SELECT SUM("Passenger Count") AS total_passengers
FROM "air_traffic"
WHERE "Year" = 2019;
  total_passengers
0         57418574


In [14]:
additional_context = ""
question = "How many passengers landed at the airport in 2019?"
a = agent.ask_question(
    question=question, 
    additional_context=additional_context, 
    distinct_char_values= False,
    trials=5
)

print(a.query)
print(a.data)


SELECT SUM("Passenger Count") FROM air_traffic WHERE "Year" = 2019 AND "Activity Type Code" = 'LANDING';
    sum
0  None


In [26]:
additional_context = ""
question = "How many passengers landed at the airport in 2019?"
a = agent.ask_question(
    question=question,
    additional_context=additional_context,
    distinct_char_values=True,
    trials=5,
)

print(a.query)
print(a.data)


Error in the query processing, trying to debug...
Trial:  5
Expecting ). Line 1, Col: 31.
  SELECT COUNT(T1.Passenger Count) FROM air_traffic AS T1 INNER JOIN air_traffic AS T2 ON T1.Year = T2.Year WHERE T1.Date >= '2019-01
Trial:  4
Expecting ). Line 1, Col: 31.
  SELECT COUNT(T1.Passenger Count) FROM air_traffic AS T1 INNER JOIN air_traffic AS T2 ON T1.Year = T2.Year WHERE T1.Date >= '2019-01
Trial:  3
Expecting ). Line 1, Col: 31.
  SELECT COUNT(T1.Passenger Count) FROM air_traffic AS T1 INNER JOIN air_traffic AS T2 ON T1.Year = T2.Year WHERE T1.Date >= '2019-01
Trial:  2
Expecting ). Line 1, Col: 31.
  SELECT COUNT(T1.Passenger Count) FROM air_traffic AS T1 INNER JOIN air_traffic AS T2 ON T1.Year = T2.Year WHERE T1.Date >= '2019-01
Trial:  1
Expecting ). Line 1, Col: 31.
  SELECT COUNT(T1.Passenger Count) FROM air_traffic AS T1 INNER JOIN air_traffic AS T2 ON T1.Year = T2.Year WHERE T1.Date >= '2019-01
Falling back to the fallback model:  ai/granite-4.0-h-micro
SELECT SUM("Passenge

In [32]:
additional_context = (
    agent.character_distinct_values_reformated
    + "The Activity Type Code field defines if a passenger departures or arrive"
)
question = "How many passengers landed at the airport in 2019?"
a = agent.ask_question(
    question=question, additional_context=additional_context, trials=5
)

print(a.query)
print(a.data)


SELECT SUM("Passenger Count") FROM air_traffic WHERE "Activity Type Code" = 'Deplaned' AND "Year" = 2019;
        sum
0  28684054


In [17]:
additional_context = "The Activity Type Code field defines if a passenger departures (e.g., Enplaned) or landed (e.g., Deplaned)"
question = "How many passengers landed at the airport in 2019?"
a = agent.ask_question(
    question=question, additional_context=additional_context, trials=3
)

print(a.query)
print(a.data)


SELECT SUM("Passenger Count") FROM air_traffic WHERE "Year" = 2019 AND "Activity Type Code" = 'Deplaned';
        sum
0  28684054


In [ ]:
additional_context = "The Activity Type Code field defines if a passenger departures (e.g., Enplaned) or landed (e.g., Deplaned)"
question = "How many passengers departed at the airport in 2019?"
a = agent.ask_question(
    question=question, additional_context=additional_context, trials=3
)

print(a.query)
print(a.data)


In [ ]:
additional_context = "The Activity Type Code field defines if a passenger departures (e.g., Enplaned) or landed (e.g., Deplaned)"
question = "How many passengers departed at the airport in 2019 via terminal 1?"
a = agent.ask_question(
    question=question, additional_context=additional_context, trials=3
)

print(a.query)
print(a.data)
